# 5.2 模型并行 (Model Parallelism)

当模型太大无法放入单GPU时，需要将模型本身拆分到多个设备上，这就是模型并行。

本节涵盖：
- 张量并行（TP）
- 流水线并行（PP）
- 序列并行（SP）

## 1. 张量并行（Tensor Parallelism）

**目的**：将单个算子拆分到多个设备

**基本原理**：将矩阵乘法按行或按列切分到不同GPU上并行计算，通过AllReduce通信同步结果。

**Megatron-LM张量并行**：
- **列并行**：将权重矩阵按列切分，每个GPU计算部分输出
- **行并行**：将权重矩阵按行切分，每个GPU处理部分输入
- 注意力头天然适合张量并行：不同头分配到不同GPU

**通信模式**：
- 前向：列并行不需要通信，行并行需要AllReduce
- 反向：对应的互补通信模式

**适用场景**：节点内（NVLink高带宽），通信延迟低

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

d_model = 256
d_ff = 1024
n_gpus = 4

W_full = torch.randn(d_model, d_ff)
x = torch.randn(8, d_model)

out_full = x @ W_full

chunk_size = d_ff // n_gpus
W_chunks = [W_full[:, i*chunk_size:(i+1)*chunk_size] for i in range(n_gpus)]

out_chunks = [x @ W_chunk for W_chunk in W_chunks]
out_tp = torch.cat(out_chunks, dim=-1)

print('=== Tensor Parallelism (Column Parallel) ===')
print(f'Full weight: {W_full.shape}')
print(f'Input: {x.shape}')
print(f'Full output: {out_full.shape}')
print(f'\nPer-GPU weight chunks: {[c.shape for c in W_chunks]}')
print(f'Per-GPU output chunks: {[c.shape for c in out_chunks]}')
print(f'\nResults match: {torch.allclose(out_full, out_tp, atol=1e-5)}')
print(f'Max difference: {(out_full - out_tp).abs().max():.8f}')

print(f'\nMemory savings per GPU:')
print(f'  Full weight: {W_full.numel() * 4 / 1e6:.2f} MB')
print(f'  Per-GPU chunk: {W_chunks[0].numel() * 4 / 1e6:.2f} MB ({1/n_gpus:.0%} of full)')
print(f'\nKey: Column parallel splits output dimension across GPUs.')
print(f'Each GPU computes a portion of the output independently.')

## 2. 流水线并行（Pipeline Parallelism）

**目的**：将模型不同层分配到不同设备

**基本原理**：将模型按层切分为多个阶段（stage），每个GPU负责一个阶段的计算，采用微批次（micro-batch）流水线使各阶段并行工作。

**流水线气泡问题**：
- 朴素流水线：GPU空闲等待时间（气泡）很大
- 微批次流水线（GPipe）：将批次拆分为多个微批次，流水线式处理
- 1F1B调度：交替前向和反向，减少峰值显存

**气泡率**：(p-1)/(m+p-1)，p为阶段数，m为微批次数

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

n_stages = 4
n_microbatches = 8

print('=== Pipeline Parallelism ===')
print(f'Stages: {n_stages}, Microbatches: {n_microbatches}')

naive_bubble = (n_stages - 1) / n_microbatches
gpipe_bubble = (n_stages - 1) / (n_microbatches + n_stages - 1)

print(f'\nBubble rate (idle GPU time):')
print(f'  Naive pipeline: {naive_bubble:.1%}')
print(f'  GPipe (micro-batch): {gpipe_bubble:.1%}')

print(f'\nPipeline schedule (1F1B, 4 stages, 4 microbatches):')
print(f'  Time  GPU0    GPU1    GPU2    GPU3')
print(f'  ----  ----    ----    ----    ----')
schedule = [
    ('F0',   'F1',   'F2',   'F3'),
    ('F1',   'F2',   'F3',   'B2'),
    ('F2',   'F3',   'B1',   'B3'),
    ('F3',   'B0',   'B2',   'idle'),
    ('B0',   'B1',   'B3',   'idle'),
    ('B1',   'B2',   'idle', 'idle'),
    ('B2',   'B3',   'idle', 'idle'),
    ('B3',   'idle', 'idle', 'idle'),
]
for t, (g0, g1, g2, g3) in enumerate(schedule):
    print(f'  t{t}    {g0:6s}  {g1:6s}  {g2:6s}  {g3:6s}')

print(f'\nF=Forward, B=Backward for micro-batch i')
print(f'1F1B alternates forward and backward to reduce peak memory.')
print(f'\nKey: More microbatches -> smaller bubble rate -> better GPU utilization.')

## 3. 序列并行（Sequence Parallelism）

**目的**：降低单层的激活值显存占用

**基本原理**：将序列维度切分到不同设备，每个设备仅处理部分token的激活值，在LayerNorm和Dropout等操作中减少重复计算。

**核心思想**：
- 注意力层：已在TP中处理，需要完整序列
- LayerNorm/Dropout：不需要跨token信息，可以按序列切分
- 将非注意力层的激活值按序列维度切分存储

**代表**：Megatron-SP

**优势**：减少激活值显存，特别对长序列训练效果显著

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

batch_size = 2
seq_len = 4096
d_model = 1024
n_gpus = 4

x = torch.randn(batch_size, seq_len, d_model)

full_ln = nn.LayerNorm(d_model)
out_full = full_ln(x)

chunk_size = seq_len // n_gpus
x_chunks = [x[:, i*chunk_size:(i+1)*chunk_size, :] for i in range(n_gpus)]
out_chunks = [full_ln(chunk) for chunk in x_chunks]
out_sp = torch.cat(out_chunks, dim=1)

full_act_mem = batch_size * seq_len * d_model * 4
sp_act_mem = full_act_mem / n_gpus

print('=== Sequence Parallelism ===')
print(f'Input: batch={batch_size}, seq_len={seq_len}, d_model={d_model}')
print(f'GPUs: {n_gpus}')
print(f'\nLayerNorm is element-wise -> can split along sequence dimension')
print(f'Results match: {torch.allclose(out_full, out_sp, atol=1e-5)}')

print(f'\nActivation memory per layer:')
print(f'  Without SP: {full_act_mem/1e6:.1f} MB')
print(f'  With SP:    {sp_act_mem/1e6:.1f} MB ({1/n_gpus:.0%} of full)')

print(f'\nWhich layers benefit from SP:')
print(f'  LayerNorm: YES (element-wise, no cross-token dependency)')
print(f'  Dropout:   YES (element-wise)')
print(f'  Attention: NO  (needs full sequence for QK computation)')
print(f'\nKey: SP reduces activation memory for non-attention layers.')
print(f'Combined with TP, it significantly reduces memory for long sequences.')